# MyDigitalTwin — Twitter/X
**Notebook 04 — Ingestion, exploration, nettoyage → Parquet**

Sources :
- `data/raw/X/data/tweets.js` → tes tweets
- `data/raw/X/data/like.js` → tweets likés

Outputs :
- `data/parquet/twitter_tweets.parquet`
- `data/parquet/twitter_likes.parquet`

## Objectifs ML
- **Clone NLP (axe 1)** : corpus de tes tweets pour TF-IDF / N-grams
- **ALS (axe 2)** : signal d'intérêt via les likes
- **K-Means (axe 3)** : activité temporelle

## 0. Initialisation Spark

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..')))
from config import RAW_DATA, WAREHOUSE

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json, re

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Twitter") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

TW_ROOT     = os.path.join(RAW_DATA, "X", "data")
PARQUET_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "data", "parquet")

def load_twitter_js(path):
    """Charge un fichier .js Twitter en retirant l'assignation window.YTD."""
    with open(path, encoding="utf-8", errors="replace") as f:
        content = f.read()
    clean = re.sub(r'^window\.YTD\.\w+\.\w+\s*=\s*', '', content.strip())
    return json.loads(clean)

Spark version : 3.5.5


26/03/28 13:31:57 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Spark version : 3.5.5


---
## PARTIE 1 — Tweets
### 1.1 Ingestion

In [3]:
raw_tweets = load_twitter_js(f"{TW_ROOT}/tweets.js")
print(f"Tweets bruts : {len(raw_tweets):,}")

def parse_tweets(raw):
    rows = []
    for entry in raw:
        t = entry.get("tweet", entry)

        text      = t.get("full_text", t.get("text", ""))
        ts        = t.get("created_at", "")
        tweet_id  = t.get("id_str", "")
        lang      = t.get("lang", "")
        rt_count  = int(t.get("retweet_count", 0) or 0)
        fav_count = int(t.get("favorite_count", 0) or 0)
        is_rt     = text.startswith("RT @")
        is_reply  = bool(t.get("in_reply_to_status_id_str"))

        # Nettoyer le texte : retirer les mentions au début des replies
        clean_text = re.sub(r'^(@\w+\s*)+', '', text).strip()

        # Extraire les URLs
        urls = [u.get("expanded_url", "") 
                for u in t.get("entities", {}).get("urls", [])]

        # Extraire les hashtags
        hashtags = [h.get("text", "").lower() 
                    for h in t.get("entities", {}).get("hashtags", [])]

        # Extraire les mentions
        mentions = [m.get("screen_name", "") 
                    for m in t.get("entities", {}).get("user_mentions", [])]

        rows.append({
            "tweet_id":     tweet_id,
            "text":         clean_text,
            "raw_text":     text,
            "lang":         lang,
            "created_at":   ts,
            "retweet_count": rt_count,
            "favorite_count": fav_count,
            "is_retweet":   is_rt,
            "is_reply":     is_reply,
            "has_media":    bool(t.get("entities", {}).get("media")),
            "hashtags":     " ".join(hashtags),
            "mentions":     " ".join(mentions),
            "urls":         " ".join(urls),
            "char_count":   len(clean_text),
            "word_count":   len(clean_text.split()),
        })
    return rows

tweet_rows = parse_tweets(raw_tweets)
print(f"Tweets parsés : {len(tweet_rows):,}")
print("Exemple :", {k: v for k, v in tweet_rows[0].items() if k in ['text', 'created_at', 'lang']})

Tweets bruts : 319
Tweets parsés : 319
Exemple : {'text': 'RT @Papino_sock: La CAF a supprimé cette vidéo maximum de Retweet  C’est nous les CHAMPIONS D’AFRIQUE 2025 🏆🇸🇳\n\n https://t.co/cIU59rvJ8Z', 'lang': 'fr', 'created_at': 'Thu Mar 19 22:56:32 +0000 2026'}


In [4]:
from datetime import datetime, timezone

def parse_twitter_date(date_str):
    if not date_str:
        return None
    try:
        dt = datetime.strptime(date_str, "%a %b %d %H:%M:%S %z %Y")
        return dt.timestamp() * 1000
    except:
        return None

# 1. D'abord remplir timestamp_ms dans les rows Python
for row in tweet_rows:
    ts_ms = parse_twitter_date(row["created_at"])
    row["timestamp_ms"] = int(ts_ms) if ts_ms else 0

# 2. Ensuite créer le DataFrame
schema_tweets = StructType([
    StructField("tweet_id",        StringType(),  True),
    StructField("text",            StringType(),  True),
    StructField("raw_text",        StringType(),  True),
    StructField("lang",            StringType(),  True),
    StructField("created_at",      StringType(),  True),
    StructField("retweet_count",   IntegerType(), True),
    StructField("favorite_count",  IntegerType(), True),
    StructField("is_retweet",      BooleanType(), True),
    StructField("is_reply",        BooleanType(), True),
    StructField("has_media",       BooleanType(), True),
    StructField("hashtags",        StringType(),  True),
    StructField("mentions",        StringType(),  True),
    StructField("urls",            StringType(),  True),
    StructField("char_count",      IntegerType(), True),
    StructField("word_count",      IntegerType(), True),
    StructField("timestamp_ms",    LongType(),    True),
])

df_tweets = spark.createDataFrame(tweet_rows, schema=schema_tweets)

# 3. Ensuite les transformations Spark
df_tweets = df_tweets.withColumn(
    "event_date", F.to_timestamp(F.col("timestamp_ms") / 1000)
)

df_tweets = df_tweets \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("platform",      F.lit("twitter")) \
    .withColumn("interaction_weight", F.lit(1.0))

print(f"Lignes : {df_tweets.count():,}")
df_tweets.select("text", "event_date", "lang", "is_retweet", "is_reply").show(5, truncate=60)

Lignes : 319
+------------------------------------------------------------+-------------------+----+----------+--------+
|                                                        text|         event_date|lang|is_retweet|is_reply|
+------------------------------------------------------------+-------------------+----+----------+--------+
|RT @Papino_sock: La CAF a supprimé cette vidéo maximum de...|2026-03-19 22:56:32|  fr|      true|   false|
|RT @Arsenal_rep1: 🚨🎙️| Neymar Jr: 🗣️\n\n"I watched som...|2026-03-19 22:54:57|  en|      true|   false|
|RT @CanalplusFoot: MAX DOWMAN DANS L'HISTOIRE DE LA PREMI...|2026-03-15 12:05:09|  fr|      true|   false|
|                                 C'est un genre de "AirDrop"|2026-03-13 01:24:57|  fr|     false|    true|
|RT @ShadesFrance: 🚨 CONCOURS\n\nGagne 2 PASS pour le jou...|2026-03-08 00:14:35|  fr|      true|   false|
+------------------------------------------------------------+-------------------+----+----------+--------+
only showing top 5 

### 1.2 Exploration

In [5]:
print("=== Répartition des types de tweets ===")
df_tweets.groupBy("is_retweet", "is_reply") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

print("\n=== Langues utilisées ===")
df_tweets.groupBy("lang").count().orderBy(F.desc("count")).show()

print("\n=== Tweets par année ===")
df_tweets.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_tweets.groupBy("event_hour").count().orderBy("event_hour").show()

=== Répartition des types de tweets ===


+----------+--------+-----+
|is_retweet|is_reply|count|
+----------+--------+-----+
|     false|    true|  180|
|      true|   false|  137|
|     false|   false|    2|
+----------+--------+-----+


=== Langues utilisées ===


+----+-----+
|lang|count|
+----+-----+
|  fr|  177|
|  en|   58|
| qme|   39|
| und|   10|
|  es|    8|
| qam|    5|
| zxx|    2|
|  in|    2|
|  tl|    2|
|  cs|    2|
|  de|    2|
|  pt|    2|
|  ht|    2|
|  lt|    2|
|  tr|    1|
|  nl|    1|
|  ro|    1|
|  sl|    1|
|  it|    1|
|  no|    1|
+----+-----+


=== Tweets par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2019|   51|
|      2020|  105|
|      2021|   39|
|      2022|   16|
|      2023|    7|
|      2024|   31|
|      2025|   55|
|      2026|   15|
+----------+-----+


=== Activité par heure ===


+----------+-----+
|event_hour|count|
+----------+-----+
|         0|    9|
|         1|    4|
|         2|    3|
|         4|    2|
|         5|    8|
|         6|    8|
|         7|    4|
|         8|    2|
|         9|    9|
|        10|   11|
|        11|    9|
|        12|    9|
|        13|    7|
|        14|    6|
|        15|    7|
|        16|   10|
|        17|   28|
|        18|   35|
|        19|   35|
|        20|   35|
+----------+-----+
only showing top 20 rows



In [6]:
print("=== Tes tweets les plus likés ===")
df_tweets.filter(~F.col("is_retweet")) \
    .orderBy(F.desc("favorite_count")) \
    .select("text", "favorite_count", "retweet_count", "event_date") \
    .limit(10) \
    .show(truncate=70)

print("\n=== Top hashtags ===")
df_tweets.filter(F.col("hashtags") != "") \
    .withColumn("hashtag", F.explode(F.split("hashtags", " "))) \
    .filter(F.col("hashtag") != "") \
    .groupBy("hashtag") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(15) \
    .show()

print("\n=== Top 20 mots (hors retweets) ===")
df_tweets.filter(~F.col("is_retweet")) \
    .withColumn("word", F.explode(F.split(F.lower("text"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").startswith("http")) \
    .filter(~F.col("word").startswith("@")) \
    .groupBy("word") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show()

print("\n=== Stats texte (hors retweets) ===")
df_tweets.filter(~F.col("is_retweet")) \
    .agg(
        F.avg("char_count").alias("avg_chars"),
        F.avg("word_count").alias("avg_words"),
        F.max("char_count").alias("max_chars"),
    ).show()

=== Tes tweets les plus likés ===
+----------------------------------------------------------------------+--------------+-------------+-------------------+
|                                                                  text|favorite_count|retweet_count|         event_date|
+----------------------------------------------------------------------+--------------+-------------+-------------------+
|                                               https://t.co/GYi9f2TZbt|           109|            0|2025-12-08 16:42:21|
|                                               https://t.co/aYtJC3Gzms|           104|            3|2021-07-08 20:28:48|
|                                                  Il reste Paris Plage|            49|            1|2020-08-14 13:00:48|
|                                               https://t.co/Oxf19MOuIQ|            22|            0|2022-04-15 05:11:20|
|                               T'aurais pu faire l'effort d'être drôle|            12|            0|2021-12-03 

+--------------+-----+
|       hashtag|count|
+--------------+-----+
| badmangangsta|    1|
|          coyg|    1|
|  mercicharlou|    1|
|honormagicbook|    1|
|   joyquestion|    1|
|      beephone|    1|
|     paris2024|    1|
|      olympics|    1|
|   noeldejumbo|    1|
+--------------+-----+


=== Top 20 mots (hors retweets) ===


+------+-----+
|  word|count|
+------+-----+
|   pas|   17|
| c'est|   15|
|   que|   15|
|   les|   14|
|  mais|   13|
|  j'ai|   12|
|   est|   11|
|   une|   11|
| comme|    9|
|  nous|    9|
|   qui|    9|
|  fait|    8|
|  dans|    8|
|  t'es|    8|
| quand|    7|
|  vous|    6|
|  pour|    6|
|   sur|    6|
|   dit|    6|
|sommes|    6|
+------+-----+


=== Stats texte (hors retweets) ===
+------------------+-----------------+---------+
|         avg_chars|        avg_words|max_chars|
+------------------+-----------------+---------+
|36.467032967032964|6.543956043956044|      273|
+------------------+-----------------+---------+



### 1.3 Écriture Parquet

In [7]:
out_tweets = f"{PARQUET_DIR}/twitter_tweets.parquet"

df_tweets.write.mode("overwrite").parquet(out_tweets)
print(f"✓ {out_tweets} — {df_tweets.count():,} lignes")

✓ ../../data/parquet/twitter_tweets.parquet — 319 lignes


---
## PARTIE 2 — Likes
### 2.1 Ingestion

In [8]:
raw_likes = load_twitter_js(f"{TW_ROOT}/like.js")
print(f"Likes bruts : {len(raw_likes):,}")

def parse_likes(raw):
    rows = []
    for entry in raw:
        lk = entry.get("like", entry)
        rows.append({
            "tweet_id":    lk.get("tweetId", ""),
            "full_text":   lk.get("fullText", ""),
            "post_url":    lk.get("expandedUrl", ""),
        })
    return rows

like_rows = parse_likes(raw_likes)
print(f"Likes parsés : {len(like_rows):,}")
print("Exemple :", like_rows[0])

Likes bruts : 68,534
Likes parsés : 68,534
Exemple : {'tweet_id': '2034953001048158579', 'full_text': 'Imagine tu portes ton enfant 9 mois et sa passion c’est ça https://t.co/pH8ZYiNTqg', 'post_url': 'https://twitter.com/i/web/status/2034953001048158579'}


In [9]:
schema_likes = StructType([
    StructField("tweet_id",  StringType(), True),
    StructField("full_text", StringType(), True),
    StructField("post_url",  StringType(), True),
])

df_likes = spark.createDataFrame(like_rows, schema=schema_likes)

df_likes = df_likes \
    .withColumn("platform",           F.lit("twitter")) \
    .withColumn("action_type",        F.lit("like")) \
    .withColumn("interaction_weight", F.lit(2.0)) \
    .withColumn("char_count",         F.length("full_text")) \
    .withColumn("word_count",         F.size(F.split(F.trim("full_text"), r"\s+")))

print(f"Lignes : {df_likes.count():,}")
df_likes.show(5, truncate=70)

Lignes : 68,534
+-------------------+----------------------------------------------------------------------+----------------------------------------------------+--------+-----------+------------------+----------+----------+
|           tweet_id|                                                             full_text|                                            post_url|platform|action_type|interaction_weight|char_count|word_count|
+-------------------+----------------------------------------------------------------------+----------------------------------------------------+--------+-----------+------------------+----------+----------+
|2034953001048158579|Imagine tu portes ton enfant 9 mois et sa passion c’est ça https://...|https://twitter.com/i/web/status/2034953001048158579| twitter|       like|               2.0|        82|        13|
|2034708788129726685|              Pourquoi y’a une PS5 au milieu ? https://t.co/IY4fGv0HZ0|https://twitter.com/i/web/status/2034708788129726685| twitte

### 2.2 Exploration

In [10]:
print("=== Nombre total de likes ===")
print(f"  {df_likes.count():,} tweets likés")

print("\n=== Top 20 mots dans les tweets likés ===")
df_likes.filter(F.col("full_text") != "") \
    .withColumn("word", F.explode(F.split(F.lower("full_text"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").startswith("http")) \
    .filter(~F.col("word").startswith("@")) \
    .groupBy("word") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show()

print("\n=== Exemples de tweets likés ===")
df_likes.select("full_text", "post_url") \
    .limit(10) \
    .show(truncate=80)

=== Nombre total de likes ===


  68,534 tweets likés

=== Top 20 mots dans les tweets likés ===


+-----------+-----+
|       word|count|
+-----------+-----+
|        les| 9596|
|       this| 9303|
|        pas| 8873|
|        que| 8101|
|       pour| 6636|
|      c’est| 6616|
|        des| 6316|
|       post| 6233|
|{learnmore}| 6157|
|        est| 6051|
|        qui| 5897|
|        une| 5398|
|       mais| 4560|
|       from| 4268|
|        sur| 4261|
|       dans| 4228|
|  suspended| 4059|
|   account.| 4057|
|       view| 4037|
|      c'est| 3482|
+-----------+-----+


=== Exemples de tweets likés ===
+--------------------------------------------------------------------------------+----------------------------------------------------+
|                                                                       full_text|                                            post_url|
+--------------------------------------------------------------------------------+----------------------------------------------------+
|Imagine tu portes ton enfant 9 mois et sa passion c’est ça https://t.co/pH8Z

### 2.3 Écriture Parquet

In [11]:
out_likes = f"{PARQUET_DIR}/twitter_likes.parquet"

df_likes.write.mode("overwrite").parquet(out_likes)
print(f"✓ {out_likes} — {df_likes.count():,} lignes")

✓ ../../data/parquet/twitter_likes.parquet — 68,534 lignes


In [13]:
spark.stop()